In [125]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import os
import h5py
from torch.utils.data import Dataset, DataLoader
from torch.autograd import Function



In [126]:
DATA_ROOT = '/kaggle/input/datasets/chandragupta0001/soli-data/dsp'



In [127]:
NUM_CLASSES  = 16
NUM_SUBJECTS = 10
FINE_GRAINED = [0, 1, 8, 9]

GESTURE_NAMES = [
    'finger_slider', 'finger_rub',  'slow_swipe',  'fast_swipe',
    'push',          'pull',         'palm_tilt',   'circle',
    'pinch_index',   'pinch_pinky',  'tap',
    'class_11',      'class_12',     'class_13',    'class_14', 'class_15'
]


In [128]:
class SoliDataset(Dataset):
    def __init__(self, data_root, subject_ids, training=False):
        self.training = training
        self.samples = []
        for fname in os.listdir(data_root):
            if not fname.endswith('.h5'):
                continue
            parts = fname.replace('.h5','').split('_')
            if int(parts[0]) in subject_ids:
                self.samples.append((
                    os.path.join(data_root, fname),
                    int(parts[0])
                ))
        print(f"Loaded {len(self.samples)} samples for subjects {subject_ids} (training={training})")

        # Per-subject normalization stats
        print("Computing per-subject normalization stats...")
        sums   = {s: 0.0 for s in subject_ids}
        sqs    = {s: 0.0 for s in subject_ids}
        counts = {s: 0   for s in subject_ids}
        for fpath, subj in self.samples:
            with h5py.File(fpath, 'r') as f:
                data = np.stack([f['ch0'][:], f['ch1'][:],
                                 f['ch2'][:], f['ch3'][:]], axis=0).astype(np.float32)
            sums[subj]   += data.sum()
            sqs[subj]    += (data ** 2).sum()
            counts[subj] += data.size
        self.subj_mean = {s: sums[s] / counts[s] for s in subject_ids}
        self.subj_std  = {s: np.sqrt(max(sqs[s]/counts[s] - self.subj_mean[s]**2, 1e-8))
                          for s in subject_ids}
        for s in subject_ids:
            print(f"  Subject {s}: mean={self.subj_mean[s]:.4f}  std={self.subj_std[s]:.4f}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        fpath, subj_id = self.samples[idx]
        with h5py.File(fpath, 'r') as f:
            data = np.stack([f['ch0'][:], f['ch1'][:],
                             f['ch2'][:], f['ch3'][:]], axis=0)   # (4, T, 1024)
            T = data.shape[1]
            if T < 30:
                pad = np.zeros((4, 30 - T, data.shape[2]), dtype=data.dtype)
                data = np.concatenate([data, pad], axis=1)
            data = data[:, :30, :].reshape(4, 30, 32, 32)
            label = int(os.path.basename(fpath).replace('.h5','').split('_')[1])

        data = data.astype(np.float32)
        data = (data - self.subj_mean[subj_id]) / self.subj_std[subj_id]

        # ── Training augmentations ──────────────────────────────────
        if self.training:
            # 1. Magnitude scaling
            data = data * np.random.uniform(0.8, 1.2)
            # 2. Gaussian noise
            data = data + np.random.normal(0, 0.1, data.shape).astype(np.float32)
            # 3. Time shift (±3 frames, wrap with zero pad)
            shift = np.random.randint(-3, 4)
            if shift != 0:
                data = np.roll(data, shift, axis=1)
                if shift > 0:
                    data[:, :shift] = 0
                else:
                    data[:, shift:] = 0
            # 4. Channel dropout (zero one channel with 30% prob)
            if np.random.rand() < 0.3:
                ch = np.random.randint(0, 4)
                data[ch] = 0

        return torch.from_numpy(data), label, subj_id

In [129]:
# ── GRL ───────────────────────────────────────────────────────────
class GradientReversalFunction(Function):
    @staticmethod
    def forward(ctx, x, lambda_):
        ctx.save_for_backward(torch.tensor(lambda_))
        return x.clone()
    @staticmethod
    def backward(ctx, grad_output):
        return -ctx.saved_tensors[0].item() * grad_output, None

class GRL(nn.Module):
    def __init__(self, lambda_=1.0):
        super().__init__()
        self.lambda_ = lambda_
    def forward(self, x):
        return GradientReversalFunction.apply(x, self.lambda_)

In [130]:
# ── Generator ─────────────────────────────────────────────────────
class Generator(nn.Module):
    def __init__(self, latent_dim=64, num_classes=NUM_CLASSES):
        super().__init__()
        self.embed   = nn.Embedding(num_classes, 16)
        self.project = nn.Sequential(
            nn.Linear(latent_dim + 16, 128 * 2 * 2 * 2), nn.ReLU()
        )
        self.upsample = nn.Sequential(
            nn.ConvTranspose3d(128, 64, 4, 2, 1), nn.BatchNorm3d(64), nn.ReLU(),
            nn.ConvTranspose3d(64,  32, 4, 2, 1), nn.BatchNorm3d(32), nn.ReLU(),
            nn.ConvTranspose3d(32,   4, 4, 2, 1), nn.Tanh(),
        )
        self.adaptive = nn.AdaptiveAvgPool3d((30, 32, 32))

    def forward(self, z, labels):
        zc = torch.cat([z, self.embed(labels)], dim=1)
        h  = self.project(zc).view(-1, 128, 2, 2, 2)
        return self.adaptive(self.upsample(h))


In [131]:

# ── Discriminator ─────────────────────────────────────────────────
class Discriminator(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv3d(4,  32, 4, 2, 1), nn.LeakyReLU(0.2),
            nn.Conv3d(32, 64, 4, 2, 1), nn.LeakyReLU(0.2),
            nn.Conv3d(64,128, 4, 2, 1), nn.LeakyReLU(0.2),
            nn.AdaptiveAvgPool3d(1), nn.Flatten(),
        )
        self.adv_head = nn.Linear(128, 1)
        self.cls_head = nn.Linear(128, num_classes)

    def forward(self, x):
        f = self.features(x)
        return self.adv_head(f), self.cls_head(f)

In [132]:

# ── Full Model ────────────────────────────────────────────────────
class FullModel(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES, num_subjects=NUM_SUBJECTS):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv3d(4,  32, 3, padding=1), nn.BatchNorm3d(32),  nn.ReLU(), nn.MaxPool3d(2),
            nn.Conv3d(32, 64, 3, padding=1), nn.BatchNorm3d(64),  nn.ReLU(), nn.MaxPool3d(2),
            nn.Conv3d(64,128, 3, padding=1), nn.BatchNorm3d(128), nn.ReLU(),
            nn.AdaptiveAvgPool3d(1), nn.Flatten(),
        )
        self.gesture_clf = nn.Sequential(
            nn.Linear(128, 64), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, num_classes),
        )
        self.grl = GRL(lambda_=1.0)
        self.domain_clf = nn.Sequential(
            nn.Linear(128, 64), nn.ReLU(),
            nn.Linear(64, num_subjects),
        )

    def forward(self, x):
        feats = self.features(x)
        return self.gesture_clf(feats), self.domain_clf(self.grl(feats))

In [133]:
# ── GAN Training ──────────────────────────────────────────────────
def train_gan(loader, G, D, opt_G, opt_D, num_epochs, device):
    G.train(); D.train()
    for epoch in range(num_epochs):
        for x_real, labels, _ in loader:
            x_real  = x_real.to(device)
            labels  = labels.to(device)
            B       = x_real.size(0)
            real_t  = torch.full((B, 1), 0.9, device=device)
            fake_t  = torch.zeros(B, 1,       device=device)

            # train D
            opt_D.zero_grad()
            v_real, c_real = D(x_real)
            loss_D = (F.binary_cross_entropy_with_logits(v_real, real_t)
                    + F.cross_entropy(c_real, labels))
            z  = torch.randn(B, 64, device=device)
            fl = torch.randint(0, NUM_CLASSES, (B,), device=device)
            v_fake, _ = D(G(z, fl).detach())
            loss_D   += F.binary_cross_entropy_with_logits(v_fake, fake_t)
            loss_D.backward()
            torch.nn.utils.clip_grad_norm_(D.parameters(), 1.0)
            opt_D.step()

            # train G twice
            for _ in range(2):
                opt_G.zero_grad()
                z  = torch.randn(B, 64, device=device)
                fl = torch.randint(0, NUM_CLASSES, (B,), device=device)
                v_fake, c_fake = D(G(z, fl))
                loss_G = (F.binary_cross_entropy_with_logits(v_fake, real_t)
                        + F.cross_entropy(c_fake, fl))
                loss_G.backward()
                torch.nn.utils.clip_grad_norm_(G.parameters(), 1.0)
                opt_G.step()

        print(f"  GAN Epoch {epoch+1:3d} | "
              f"D: {loss_D.item():.4f} | G: {loss_G.item():.4f}")


In [134]:
# ── DANN Training ─────────────────────────────────────────────────
def train_dann(loader, model, optimizer, num_epochs, device):
    for epoch in range(num_epochs):
        model.train()
        p = epoch / num_epochs
        lam = (2.0 / (1.0 + np.exp(-10 * p)) - 1.0) * 0.1   # cap at 0.1
        model.grl.lambda_ = lam
        correct = total = g_sum = d_sum = 0

        for x, labels, subjects in loader:
            x, labels, subjects = (x.to(device),
                                   labels.to(device),
                                   subjects.to(device))
            optimizer.zero_grad()
            g_logits, d_logits = model(x)
            loss_g = F.cross_entropy(g_logits, labels)
            loss_d = F.cross_entropy(d_logits, subjects)
            (loss_g + loss_d).backward()
            optimizer.step()
            correct += (g_logits.argmax(1) == labels).sum().item()
            total   += len(labels)
            g_sum   += loss_g.item()
            d_sum   += loss_d.item()

        print(f"  Epoch {epoch+1:3d} | "
              f"Gest: {g_sum/len(loader):.4f} | "
              f"Dom: {d_sum/len(loader):.4f} | "
              f"Acc: {correct/total:.4f} | λ: {lam:.3f}")


In [135]:
# ── Evaluation ────────────────────────────────────────────────────
def evaluate(model, loader, device):
    model.eval()
    correct_per = [0] * NUM_CLASSES
    total_per   = [0] * NUM_CLASSES

    with torch.no_grad():
        for x, labels, _ in loader:
            x, labels = x.to(device), labels.to(device)
            logits, _ = model(x)
            preds = logits.argmax(1)
            for p, l in zip(preds, labels):
                total_per[l]   += 1
                if p == l:
                    correct_per[l] += 1

    print(f"\n{'Gesture':<20} {'Acc':>8}")
    print("-" * 30)
    fg_accs = []
    for i in range(NUM_CLASSES):
        acc = correct_per[i] / max(total_per[i], 1)
        tag = " ← FG" if i in FINE_GRAINED else ""
        print(f"{GESTURE_NAMES[i]:<20} {acc:>8.4f}{tag}")
        if i in FINE_GRAINED:
            fg_accs.append(acc)

    overall = sum(correct_per) / max(sum(total_per), 1)
    print(f"\nOverall Accuracy:      {overall:.4f}")
    print(f"Fine-Grained Accuracy: {np.mean(fg_accs):.4f}")
    return overall, np.mean(fg_accs)


In [136]:
# ── Augmented Dataset ─────────────────────────────────────────────
class AugmentedDataset(Dataset):
    def __init__(self, real_dataset, synthetic_samples):
        self.samples = []
        for tensor, label, subj in real_dataset:
            self.samples.append((tensor, label, subj))
        for arr, label, _ in synthetic_samples:
            self.samples.append((torch.from_numpy(arr), label, 0))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        tensor, label, subj = self.samples[idx]
        return tensor, label, subj

In [137]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device}")

# ── data ──────────────────────────────────────────────────────────
train_dataset = SoliDataset(DATA_ROOT, subject_ids=[5,6,7,8,9], training=True)
test_dataset  = SoliDataset(DATA_ROOT, subject_ids=[0,1,2,3,4], training=False)
train_loader  = DataLoader(train_dataset, batch_size=16,
                           shuffle=True,  num_workers=2)
test_loader   = DataLoader(test_dataset,  batch_size=16,
                           shuffle=False, num_workers=2)

# ── phase 1: GAN ──────────────────────────────────────────────────
print("\nPhase 1: Training GAN (50 epochs)...")
G     = Generator().to(device)
D     = Discriminator().to(device)
opt_G = torch.optim.Adam(G.parameters(), lr=2e-4, betas=(0.5,0.999))
opt_D = torch.optim.Adam(D.parameters(), lr=5e-5, betas=(0.5,0.999))
train_gan(train_loader, G, D, opt_G, opt_D, num_epochs=50, device=device)

# ── phase 2: generate synthetic samples ───────────────────────────
print("\nGenerating synthetic fine-grained samples...")
G.eval()
synthetic = []
with torch.no_grad():
    for class_id in FINE_GRAINED:
        z  = torch.randn(100, 64, device=device)
        fl = torch.full((100,), class_id, dtype=torch.long, device=device)
        fake = G(z, fl).cpu()
        for i in range(100):
            synthetic.append((fake[i].numpy(), class_id, -1))
print(f"Generated {len(synthetic)} synthetic samples")

# ── phase 3: augment + retrain ────────────────────────────────────
aug_dataset = AugmentedDataset(train_dataset, synthetic)
aug_loader  = DataLoader(aug_dataset, batch_size=16,
                         shuffle=True, num_workers=2)
print(f"Augmented size: {len(aug_dataset)}")

print("\nPhase 2: Training DANN Classifier (50 epochs)...")
model     = FullModel().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
train_dann(aug_loader, model, optimizer, num_epochs=80, device=device)

# ── evaluate ──────────────────────────────────────────────────────
print("\nFold 0 Results:")
fold0_overall, fold0_fg = evaluate(model, test_loader, device)

Device: cuda
Loaded 2375 samples for subjects [5, 6, 7, 8, 9] (training=True)
Computing per-subject normalization stats...
  Subject 5: mean=0.0045  std=0.0550
  Subject 6: mean=0.0048  std=0.0567
  Subject 7: mean=0.0051  std=0.0576
  Subject 8: mean=0.0056  std=0.0615
  Subject 9: mean=0.0048  std=0.0570
Loaded 2375 samples for subjects [0, 1, 2, 3, 4] (training=False)
Computing per-subject normalization stats...
  Subject 0: mean=0.0048  std=0.0568
  Subject 1: mean=0.0060  std=0.0631
  Subject 2: mean=0.0049  std=0.0575
  Subject 3: mean=0.0042  std=0.0537
  Subject 4: mean=0.0047  std=0.0565

Phase 1: Training GAN (50 epochs)...
  GAN Epoch   1 | D: 3.4245 | G: 4.2883
  GAN Epoch   2 | D: 3.1798 | G: 5.7534
  GAN Epoch   3 | D: 2.8625 | G: 5.8514
  GAN Epoch   4 | D: 2.6668 | G: 5.3363
  GAN Epoch   5 | D: 2.7271 | G: 6.2586
  GAN Epoch   6 | D: 2.6543 | G: 5.4814
  GAN Epoch   7 | D: 2.1814 | G: 5.3443
  GAN Epoch   8 | D: 2.5944 | G: 5.2408
  GAN Epoch   9 | D: 2.0442 | G: 5.772

In [138]:
import torch, os
os.makedirs('/kaggle/working/checkpoints', exist_ok=True)
torch.save(model.state_dict(), '/kaggle/working/checkpoints/model_final.pth')
print("Saved to /kaggle/working/checkpoints/model_final.pth")

Saved to /kaggle/working/checkpoints/model_final.pth


In [140]:
torch.save(G.state_dict(), '/kaggle/working/checkpoints/generator.pth')

In [141]:
model = FullModel().to(device)
model.load_state_dict(torch.load('model_final.pth'))
model.eval()

FileNotFoundError: [Errno 2] No such file or directory: 'model_final.pth'